<a href="https://colab.research.google.com/github/YukthiN/W8-Chatbot-DOC/blob/main/rag_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Simple RAG Pipeline — No LangGraph
```
PDF → Extract → Chunk → Embed → FAISS → Retrieve → Generate → Answer
```
**Runtime:** GPU → Runtime → Change runtime type → T4 GPU

## 📦 Cell 1 — Install

In [ ]:
!pip install -q \
    pymupdf \
    langchain \
    langchain-community \
    langchain-text-splitters \
    sentence-transformers \
    faiss-cpu \
    transformers \
    accelerate \
    bitsandbytes

print('✅ All packages installed')

## 🔧 Cell 2 — All Imports + GPU Check

In [ ]:
# ── Standard ──────────────────────────────────────────────────────────────────
import os
import re
import numpy as np
from typing import List

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch

# ── PyMuPDF ───────────────────────────────────────────────────────────────────
import fitz

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# ── Embeddings ────────────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

# ── FAISS ─────────────────────────────────────────────────────────────────────
import faiss

# ── HuggingFace ───────────────────────────────────────────────────────────────
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

# ── GPU Check ─────────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ All imports done')
print(f'   Device : {device}')
if device == 'cuda':
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('   ⚠️  No GPU — switch to T4 in Runtime settings')

## ⚙️ Cell 3 — Config

In [ ]:
# ── Chunking ──────────────────────────────────────────────────────────────────
CHUNK_SIZE    = 400   # characters per chunk
CHUNK_OVERLAP = 50    # overlap between chunks

# ── Embedding ──────
EMBED_MODEL   = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIM     = 384

# ── Retrieval ──────
TOP_K         = 3
SCORE_THRESH  = 0.15

# ── Generator ──────
GEN_MODEL      = 'mistralai/Mistral-7B-Instruct-v0.2'
MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.3

print('✅ Config ready')

## 📄 Cell 4 — Upload & Extract PDF

In [ ]:
from google.colab import files

print('📂 Upload your PDF...')
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f'✅ Uploaded: {pdf_path}')


def extract_pdf_text(path: str) -> List[Document]:
    """Extract text page by page using PyMuPDF."""
    doc = fitz.open(path)
    documents = []
    for page_num, page in enumerate(doc):
        text = page.get_text('text').strip()
        if not text:
            continue
        text = re.sub(r'\n{3,}', '\n\n', text)  # clean extra blank lines
        documents.append(Document(
            page_content=text,
            metadata={'source': path, 'page': page_num + 1}
        ))
    doc.close()
    return documents


raw_docs = extract_pdf_text(pdf_path)

print(f'\n📖 PDF Stats:')
print(f'   Pages   : {len(raw_docs)}')
print(f'   Chars   : {sum(len(d.page_content) for d in raw_docs):,}')
print(f'\n--- Page 1 Preview ---')
print(raw_docs[0].page_content[:400])

## ✂️ Cell 5 — Chunk

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)

chunks = splitter.split_documents(raw_docs)
chunk_texts = [c.page_content for c in chunks]

print(f'✅ Chunking done')
print(f'   Total chunks : {len(chunks)}')
print(f'   Avg length   : {np.mean([len(t) for t in chunk_texts]):.0f} chars')
print(f'\n--- Sample Chunk ---')
print(chunk_texts[0])

## 🧬 Cell 6 — Embed (BERT-style)

In [ ]:
print(f'Loading embedding model: {EMBED_MODEL}...')
embed_model = SentenceTransformer(EMBED_MODEL, device=device)

embeddings = embed_model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,   # normalise → dot product = cosine similarity
    convert_to_numpy=True
).astype('float32')              # FAISS requires float32

print(f'\n✅ Embeddings done')
print(f'   Shape  : {embeddings.shape}')   # (n_chunks, 384)
print(f'   Memory : {embeddings.nbytes / 1e6:.2f} MB')

## 🗃️ Cell 7 — FAISS Index

In [ ]:
# IndexFlatIP = exact inner product search on normalised vectors = cosine similarity
index = faiss.IndexFlatIP(EMBED_DIM)
index.add(embeddings)

print(f'✅ FAISS index built')
print(f'   Vectors stored : {index.ntotal}')


def retrieve(query: str, top_k: int = TOP_K, threshold: float = SCORE_THRESH) -> List[dict]:
    """Embed query → search FAISS → filter by threshold → return chunks."""

    # 1. Embed query
    q_vec = embed_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype('float32')

    # 2. Search
    scores, indices = index.search(q_vec, top_k)
    scores, indices = scores[0], indices[0]

    # 3. Filter
    results = []
    for score, idx in zip(scores, indices):
        if idx == -1 or score < threshold:
            continue
        results.append({
            'text'    : chunks[idx].page_content,
            'score'   : float(score),
            'page'    : chunks[idx].metadata.get('page', '?')
        })
    return results


# Quick test
test = retrieve('What is this paper about?')
print(f'\n🔍 Test retrieval → {len(test)} chunks found')
for i, r in enumerate(test):
    print(f'  [{i+1}] Score={r["score"]:.3f} | Page {r["page"]}')
    print(f'       {r["text"][:150]}...')

In [ ]:
# ─── Debug: Check raw FAISS scores ───────────────────────────────────────────
query = 'What is this paper about?'

q_vec = embed_model.encode(
    [query],
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')

scores, indices = index.search(q_vec, 5)

print('Raw scores from FAISS:')
print(scores[0])
print('\nIndices:')
print(indices[0])
print('\nSample chunk text at index 0:')
print(chunks[0].page_content[:200])

## 🤖 Cell 8 — Load Generator (Mistral-7B, 4-bit)

In [ ]:
# 4-bit quantization — compresses 7B model from ~14GB → ~4GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

print(f'Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print(f'Loading model (may take 3-5 min)...')
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
)
gen_model.eval()

# ✅ task='text-generation' — correct for GPT/Mistral (causal LM)
# ❌ NOT 'summarization' — that's only for BART (encoder-decoder)
gen_pipe = pipeline(
    task='text-generation',
    model=gen_model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False,
)

print('\n✅ Generator ready')

## 🔗 Cell 9 — RAG Function

In [ ]:
def ask(question: str) -> str:
    """
    Full RAG pipeline in one function:
      1. Retrieve relevant chunks from FAISS
      2. Build prompt with context
      3. Generate answer with Mistral
    """

    # ── Step 1: Retrieve ──────────────────────────────────────────────────────
    results = retrieve(question)

    if not results:
        return (
            f"I couldn't find relevant information about '{question}' "
            f"in the document. Try rephrasing your question."
        )

    # ── Step 2: Build context ─────────────────────────────────────────────────
    context = '\n\n---\n\n'.join(
        f"[Page {r['page']} | Score {r['score']:.2f}]\n{r['text']}"
        for r in results
    )

    # ── Step 3: Build prompt ──────────────────────────────────────────────────
    # Mistral Instruct format: [INST] ... [/INST]
    prompt = f"""[INST] You are a helpful assistant. Answer the question using ONLY the provided context.
If the context doesn't contain enough information, say so honestly.

Context:
{context}

Question: {question} [/INST]"""

    # ── Step 4: Generate ──────────────────────────────────────────────────────
    output = gen_pipe(prompt)[0]['generated_text']

    return output.strip()


print('✅ ask() function ready')
print('   Usage: ask("your question here")')

## 🚀 Cell 10 — Ask a Question

In [ ]:
question = 'What is the main contribution of this paper?'  # ← change this

print(f'❓ {question}')
print('=' * 60)

answer = ask(question)

print(answer)
print('=' * 60)

# Show sources
sources = retrieve(question)
print('\n📚 Sources:')
for i, r in enumerate(sources):
    print(f'  [{i+1}] Page {r["page"]} | Score {r["score"]:.3f}')

## 💬 Cell 11 — Interactive Q&A Loop

In [ ]:
print('🤖 RAG ready! Type your question (or "quit" to stop)\n')

while True:
    try:
        question = input('You: ').strip()
    except EOFError:
        break

    if not question:
        continue
    if question.lower() in ('quit', 'exit', 'q'):
        print('Bye!')
        break

    answer  = ask(question)
    sources = retrieve(question)

    print(f'\n🤖 {answer}')
    if sources:
        pages = [str(r['page']) for r in sources]
        print(f'   (Pages: {", ".join(pages)})')
    print('-' * 50 + '\n')